In [ ]:
import pandas as pd

ALL FILES LOAD + MERGE (MAIN CODE)

In [ ]:
import pandas as pd

# load files
udemy = pd.read_csv("/content/Udemy.csv")
edx = pd.read_csv("/content/edx.csv")
skillshare = pd.read_csv("/content/skillshare.csv")
coursera = pd.read_csv("/content/Coursera.csv")

def clean(df):
    # safe check (important)
    if "title" not in df.columns:
        return None

    df["subject"] = "Online Course"
    df["topic"] = df["title"]
    df["type"] = "Course"

    # default values if missing
    if "level" not in df.columns:
        df["level"] = "Beginner"

    if "link" not in df.columns:
        df["link"] = "https://www.google.com"

    return df[["subject", "topic", "level", "type", "title", "link"]]

datasets = [udemy, edx, skillshare, coursera]

cleaned = []

for d in datasets:
    c = clean(d)
    if c is not None:
        cleaned.append(c)

final_df = pd.concat(cleaned, ignore_index=True)

final_df.to_csv("final_dataset.csv", index=False)

print("✅ Dataset ready!")
final_df.head()

✅ Dataset ready!


,subject,topic,level,type,title,link
0,Online Course,The Complete Python Bootcamp From Zero to Hero...,All Levels,Course,The Complete Python Bootcamp From Zero to Hero...,https://www.google.com
1,Online Course,The Complete 2023 Web Development Bootcamp,All Levels,Course,The Complete 2023 Web Development Bootcamp,https://www.google.com
2,Online Course,The Web Developer Bootcamp 2023,All Levels,Course,The Web Developer Bootcamp 2023,https://www.google.com
3,Online Course,100 Days of Code: The Complete Python Pro Boot...,All Levels,Course,100 Days of Code: The Complete Python Pro Boot...,https://www.google.com
4,Online Course,React - The Complete Guide 2023 (incl. React R...,All Levels,Course,React - The Complete Guide 2023 (incl. React R...,https://www.google.com


In [ ]:
from google.colab import files
files.download("final_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

df = pd.read_csv("/content/final_dataset.csv")
df.head()

,subject,topic,level,type,title,link
0,Online Course,The Complete Python Bootcamp From Zero to Hero...,All Levels,Course,The Complete Python Bootcamp From Zero to Hero...,https://www.google.com
1,Online Course,The Complete 2023 Web Development Bootcamp,All Levels,Course,The Complete 2023 Web Development Bootcamp,https://www.google.com
2,Online Course,The Web Developer Bootcamp 2023,All Levels,Course,The Web Developer Bootcamp 2023,https://www.google.com
3,Online Course,100 Days of Code: The Complete Python Pro Boot...,All Levels,Course,100 Days of Code: The Complete Python Pro Boot...,https://www.google.com
4,Online Course,React - The Complete Guide 2023 (incl. React R...,All Levels,Course,React - The Complete Guide 2023 (incl. React R...,https://www.google.com


In [ ]:
df["link"] = "https://www.google.com/search?q=" + df["title"].astype(str).str.replace(" ", "+")

In [ ]:
import re

df["clean_title"] = df["title"].astype(str).apply(lambda x: re.sub(r'[^a-zA-Z0-9 ]', '', x))
df["link"] = "https://www.google.com/search?q=" + df["clean_title"].str.replace(" ", "+")

In [ ]:
df.to_csv("final_dataset_fixed.csv", index=False)

In [ ]:
from google.colab import files
files.download("final_dataset_fixed.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install flask flask-cors

In [ ]:
import pandas as pd

df = pd.read_csv("/content/final_dataset_fixed.csv")

df["content"] = df["title"].astype(str) + " " + df["level"].astype(str)

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("/content/final_dataset_fixed.csv")

df["content"] = df["title"].astype(str) + " " + df["level"].astype(str)

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df["content"])

In [ ]:
def recommend(query):
    query_vec = vectorizer.transform([query])
    similarity = cosine_similarity(query_vec, tfidf_matrix)

    results = similarity[0].argsort()[::-1][:5]

    return df.iloc[results][["title", "level", "link"]]

In [ ]:
print(recommend("machine learning beginner"))

                                                  title     level  \
4276  Machine Learning Guide: Learn Machine Learning...  Beginner   
3865  Machine Learning In The Cloud With Azure Machi...  Beginner   
4056  Machine Learning : Complete Maths for Machine ...  Beginner   
8037                          Machine Learning Tutorial  Beginner   
656                    Machine Learning with Javascript  Beginner   

                                                   link  
4276  https://www.google.com/search?q=Machine+Learni...  
3865  https://www.google.com/search?q=Machine+Learni...  
4056  https://www.google.com/search?q=Machine+Learni...  
8037  https://www.google.com/search?q=Machine+Learni...  
656   https://www.google.com/search?q=Machine+Learni...  


Create Flask backend

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

Add your model inside

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("final_dataset_fixed.csv")

df["content"] = df["title"].astype(str) + " " + df["level"].astype(str)

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df["content"])

Recommendation function

In [ ]:
def recommend(query):
    query_vec = vectorizer.transform([query])
    similarity = cosine_similarity(query_vec, tfidf_matrix)

    results = similarity[0].argsort()[::-1][:5]

    return df.iloc[results][["title", "level", "link"]].to_dict(orient="records")

API route

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

app = Flask(__name__)
CORS(app)

# Load dataset
df = pd.read_csv("final_dataset_fixed.csv")

# Create content
df["content"] = df["title"].astype(str) + " " + df["level"].astype(str)

# Model
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df["content"])

# Recommendation function
def recommend(query):
    query_vec = vectorizer.transform([query])
    similarity = cosine_similarity(query_vec, tfidf_matrix)

    results = similarity[0].argsort()[::-1][:5]

    return df.iloc[results][["title", "level", "link"]].to_dict(orient="records")

# API route
@app.route("/recommend", methods=["POST"])
def get_recommendations():
    data = request.json
    query = data.get("query")

    result = recommend(query)

    return jsonify(result)

# RUN SERVER (VERY IMPORTANT)
if __name__ == "__main__":
    app.run(debug=True, port=5001)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)
